# EMNIST Balanced Model Development and Validation — Reproducibility Notebook

**Dissertation:** *Machine Learning on the Edge: Performance and Security Evaluation of Artificial Neural Network Implementations in Embedded Systems*

**Branch:** `EMNIST-Models-Data-and-Notebook`  
**Related code branch:** `EMNIST-Models-Code`

This notebook documents the **existing experimental evidence** for EMNIST Balanced dataset verification, fully connected model development, CNN model comparisons, selection of the compact CNN, and preliminary quantization comparisons. It reads the original exported files; **it does not retrain models, recalculate historical test accuracies, or perform FPGA/Unity inference**.

The source data and screenshots are stored separately from training scripts to preserve the original experimental workflow. All reported accuracy values are derived from saved summaries or from integer correct/total counts in those summaries. Any missing files are identified rather than silently replaced.

## 1. Evidence inventory and setup

The GitHub branch screenshot shows files stored in its **root directory**. The notebook searches both its own directory and an optional `data/` directory, so files can remain flat or be reorganized later. It uses only **NumPy, pandas, and Matplotlib** for analysis; PyTorch is optional and is used solely when checkpoint metadata is examined.

Original screenshots, such as `emnist_cnn_train_result1.png`, are supporting evidence, not a substitute for exported JSON or CSV results.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image, Markdown

ROOT = Path.cwd()
SEARCH_ROOTS = [ROOT, ROOT / "data", ROOT / "results", ROOT.parent]

def locate(*names):
    for name in names:
        for folder in SEARCH_ROOTS:
            candidate = folder / name
            if candidate.is_file():
                return candidate
    return None

def load_json(*names):
    path = locate(*names)
    if path is None:
        print("Not found:", " / ".join(names))
        return None
    with path.open(encoding="utf-8") as f:
        return json.load(f)

def show_image(*names, width=850):
    path = locate(*names)
    if path:
        display(Image(filename=str(path), width=width))
    else:
        print("Optional image not found:", " / ".join(names))

print("Notebook location:", ROOT)
print("Data roots searched:", *SEARCH_ROOTS, sep="\n  - ")

In [ ]:
EXPECTED = {
    "Dataset verification": ["EMNIST_Balanced_Verification_Summary.txt", "EMNIST_Balanced_Orientation_Check.png", "EMNIST_Balanced_Reference_Vectors.npz"],
    "Dense model reports": ["EMNIST_Balanced_baseline_Training_Summary.json", "EMNIST_Balanced_expanded_Training_Summary.json", "training_summary_mlp200.json"],
    "CNN model reports": ["training_summary.json", "training_summary_improved.json", "training_summary_compact.json"],
    "CNN validation histories": ["validation_history.csv", "validation_history_compact.csv"],
    "CNN confusion matrices": ["test_confusion_matrix.csv", "test_confusion_matrix_compact.csv"],
    "Model comparisons": ["quantization_comparison.csv"],
    "Selected model": ["best_cnn_checkpoint_compact.pt"],
}
records = []
for group, names in EXPECTED.items():
    for name in names:
        p = locate(name)
        records.append({"Category": group, "Filename": name, "Found": p is not None,
                        "Size (KiB)": round(p.stat().st_size / 1024, 1) if p else np.nan})
inventory = pd.DataFrame(records)
display(inventory)
print(f"Located {inventory['Found'].sum()} of {len(inventory)} listed evidence files.")

## 2. EMNIST Balanced verification

The official [NIST EMNIST dataset](https://www.nist.gov/itl/products-and-services/emnist-dataset) supplies 112,800 training images and 18,800 independent test images. The Balanced split contains 47 classes with 2,400 training images and 400 test images per class. Each image is 28 × 28 grayscale.

The raw IDX image representation requires a **single transpose** for upright display and consistent row-major input preparation. The verification artifacts document the original dataset checks. The dataset itself is not committed to this branch.

In [ ]:
verification = locate("EMNIST_Balanced_Verification_Summary.txt")
if verification:
    print(verification.read_text(encoding="utf-8"))
else:
    print("Verification summary unavailable; see EMNIST-Models-Code/emnist_balanced_verify.py")
show_image("EMNIST_Balanced_Orientation_Check.png", width=900)
reference_path = locate("EMNIST_Balanced_Reference_Vectors.npz")
if reference_path:
    with np.load(reference_path, allow_pickle=False) as vectors:
        print("Reference vector arrays:", {key: vectors[key].shape for key in vectors.files})

## 3. Fully connected network experiments

The earlier EMNIST model development adapted the MNIST fully connected workflow to 47 outputs and explored larger hidden layers. These results explain why CNN alternatives were evaluated. They are **software model-development results**, not FPGA performance measurements.

The archived network2-compatible weights and biases provide additional reproducibility evidence; this notebook does not deserialize or execute those large text exports.

In [ ]:
dense_sources = [
    ("Baseline fully connected", ["EMNIST_Balanced_baseline_Training_Summary.json"]),
    ("Expanded 784–64–32–47", ["EMNIST_Balanced_expanded_Training_Summary.json"]),
    ("MLP 784–200–47", ["training_summary_mlp200.json"]),
]

def accuracy(row, prefix):
    for key in (prefix + "_accuracy", "test_accuracy_for_best_checkpoint" if prefix == "test" else "best_validation_accuracy"):
        if key in row and isinstance(row[key], (float, int)):
            return float(row[key])
    return np.nan

def summary_row(name, summary):
    if not summary:
        return None
    test_n = summary.get("test_total", summary.get("test_count", summary.get("official_test_count")))
    test_correct = summary.get("test_correct", summary.get("test_correct_for_best_checkpoint"))
    acc = summary.get("test_accuracy", summary.get("test_accuracy_for_best_checkpoint"))
    if acc is None and test_n and test_correct is not None:
        acc = test_correct / test_n
    val = summary.get("best_validation_accuracy")
    if val is None and summary.get("best_validation_total") and summary.get("best_validation_correct") is not None:
        val = summary["best_validation_correct"] / summary["best_validation_total"]
    return {"Experiment":name, "Architecture": str(summary.get("architecture", "See training script")),
            "Training n":summary.get("training_count"), "Validation n":summary.get("validation_count"),
            "Best validation (%)":100 * val if val is not None else np.nan,
            "Test correct":test_correct, "Test n":test_n,
            "Test accuracy (%)":100 * acc if acc is not None else np.nan,
            "Best epoch":summary.get("best_validation_epoch", summary.get("best_validation_epoch_zero_based")),
            "Source": "saved JSON"}

dense_rows=[]
for label, names in dense_sources:
    row=summary_row(label, load_json(*names))
    if row: dense_rows.append(row)
if dense_rows: display(pd.DataFrame(dense_rows).round(3))
else: print("Dense training summaries not found at the paths searched.")

### Preserved dense-network weight exports

The branch inventory also contains `EMNIST_Balanced_baseline_WeightsAndBiases...`, `EMNIST_Balanced_expanded_Best_WeightsAndBiases...`, `EMNIST_Balanced_expanded_Final_WeightsAndBiases...`, and the MLP-200 best/final weights files. Best and final exports are distinguished because they need not represent the same training epoch. This notebook treats those as **archived model artifacts**, not interchangeable checkpoints.

In [ ]:
exports = sorted({p.name for d in SEARCH_ROOTS if d.exists() for p in d.glob("*WeightsAndBiases*")})
print("Archived dense weight/bias exports:")
for filename in exports: print(" •", filename)
if not exports: print("No dense-network text exports found in the searched directories.")

## 4. CNN reference, improved CNN, and compact CNN

Three CNN experiments were used to assess the relationship between character classification performance and model complexity:

- **Reference CNN**: initial four-convolution-layer software reference.
- **Improved CNN**: larger four-convolution-layer model with adjusted regularization.
- **Compact CNN**: three convolutional layers (16, 32, 64 channels), two 2 × 2 max-pooling stages, global average pooling, and a 47-class output layer.

The compact CNN contains **26,351 trainable parameters** and was selected for subsequent FPGA and Unity adaptation. The original test split is independent of the training/validation partition.

In [ ]:
cnn_sources = [
    ("CNN reference", "training_summary.json"),
    ("CNN improved", "training_summary_improved.json"),
    ("CNN compact (selected)", "training_summary_compact.json"),
]
cnn_summaries = {label: load_json(filename) for label, filename in cnn_sources}
cnn_rows = [summary_row(label, data) for label, data in cnn_summaries.items() if data]
cnn_table = pd.DataFrame(cnn_rows)
if not cnn_table.empty:
    display(cnn_table.round(3))
    available = cnn_table.dropna(subset=["Test accuracy (%)"])
    if not available.empty:
        fig, ax = plt.subplots(figsize=(8, 3.8))
        ax.barh(available["Experiment"], available["Test accuracy (%)"])
        ax.set_xlabel("Independent test-set accuracy (%)")
        ax.set_title("EMNIST Balanced CNN software reference accuracy")
        ax.invert_yaxis()
        for i, val in enumerate(available["Test accuracy (%)"]):
            ax.text(val + .15, i, f"{val:.2f}%", va="center")
        ax.set_xlim(0, max(100, available["Test accuracy (%)"].max() + 7))
        plt.tight_layout(); plt.show()
else: print("CNN JSON summaries are not present in the searched locations.")

### Reference results used in the dissertation

The original saved JSON reports for the selected model record **9,825 / 11,280 validation classifications (87.10%) at epoch 36** and **16,327 / 18,800 independent test classifications (86.85%)**. The improved CNN records **16,820 / 18,800 (89.47%)** on the test set. These are *software reference results*, not accuracy measurements on the ZedBoard.

The next cell checks those figures against the original summaries when present, instead of assuming the copied values are correct.

In [ ]:
EXPECTED_COMPACT = {"best_validation_correct": 9825, "best_validation_total": 11280,
                    "best_validation_epoch": 36, "test_correct": 16327, "test_total": 18800,
                    "parameter_count": 26351}
compact = cnn_summaries.get("CNN compact (selected)")
if compact:
    checks = [{"Measurement":k, "Expected from archived result":v,
               "Loaded from JSON":compact.get(k), "Matches":compact.get(k)==v}
              for k,v in EXPECTED_COMPACT.items()]
    display(pd.DataFrame(checks))
else: print("The compact model summary is missing; verification deferred until the JSON is present.")

## 5. Validation history

The validation-history CSV files show accuracy and loss by epoch. The plotted series are loaded from the original files; the number of epochs and the corresponding learning-rate schedule are not inferred from screenshot labels.

In [ ]:
histories = [
    ("CNN improved", "validation_history.csv"),
    ("CNN compact", "validation_history_compact.csv"),
    ("MLP 200", "validation_history_mlp200.csv"),
]
fig, ax = plt.subplots(figsize=(9,4.5)); plotted=0
for label, filename in histories:
    path=locate(filename)
    if path is None: continue
    df=pd.read_csv(path)
    ycol=next((x for x in ["validation_accuracy", "val_accuracy"] if x in df),None)
    if ycol is None or "epoch" not in df: continue
    values=pd.to_numeric(df[ycol],errors="coerce")
    ax.plot(df["epoch"], values*100 if values.max() <= 1 else values, label=label)
    plotted += 1
if plotted:
    ax.set(title="Validation accuracy by training epoch",xlabel="Epoch",ylabel="Validation accuracy (%)")
    ax.legend();ax.grid(alpha=.25);plt.tight_layout();plt.show()
else:
    plt.close(fig)
    print("No compatible validation-history CSV files found.")

## 6. Class-level classification performance

Confusion matrices are used to examine class-level errors on the official test set. CSV matrices have **no column header** and should contain 47 rows and 47 columns. Matrix row totals represent examples in the true class; columns represent predicted classes. The class-level visualization below is generated from the saved counts, not estimated from screenshots.

In [ ]:
confusions = [("Reference CNN", "test_confusion_matrix.csv"),
              ("Improved CNN", "test_confusion_matrix_improved.csv"),
              ("Compact CNN", "test_confusion_matrix_compact.csv")]
for label, filename in confusions:
    p=locate(filename)
    if not p: continue
    arr=pd.read_csv(p,header=None).to_numpy()
    if arr.shape != (47,47):
        print(f"{filename}: unexpected matrix shape {arr.shape}; not plotted")
        continue
    print(f"{label}: n={arr.sum():,}, correct={np.trace(arr):,}, accuracy={np.trace(arr)/arr.sum():.2%}")
    fig, ax=plt.subplots(figsize=(6.8,5.5))
    im=ax.imshow(arr,aspect="auto")
    ax.set(title=f"{label} — test confusion matrix",xlabel="Predicted class ID",ylabel="True class ID")
    fig.colorbar(im,ax=ax,label="Image count")
    plt.tight_layout();plt.show()

## 7. Preliminary quantization comparison

The model-selection work also compared CNN performance after simulated weight and activation quantization. **Fake quantization is not FPGA integer inference:** the comparison script keeps accumulation and biases in floating-point. The results are useful for evaluating sensitivity to lower-precision representations but do not establish numerical equivalence with RTL. Actual integer arithmetic tests, RTL vectors, board verification, and timing/power measurements belong in the FPGA implementation branches.

In [ ]:
quant_path=locate("quantization_comparison.csv")
if quant_path:
    quant=pd.read_csv(quant_path)
    display(quant)
    print("CSV columns:",list(quant.columns))
else:
    print("quantization_comparison.csv not found.")
show_image("emnist_quantization_comparison.png",width=850)

## 8. Selected model checkpoint and transfer to implementation branches

The final compact CNN's trained parameters are archived as `best_cnn_checkpoint_compact.pt`. This checkpoint is the **software reference artifact** for later FPGA parameter conversion and Unity deployment. The other CNN checkpoints support comparisons between architectures.

Checkpoint contents should be examined without retraining. The code below attempts a safe PyTorch metadata read (`weights_only=True`) and reports tensor shapes, total parameters, and selected scalar metadata. It intentionally does **not** run the model. Loading checkpoint files should be limited to trusted repository artifacts.

In [ ]:
checkpoint_files = ["best_cnn_checkpoint.pt", "best_cnn_checkpoint_improved.pt",
                    "best_cnn_checkpoint_compact.pt", "best_mlp_checkpoint.pt", "final_mlp_checkpoint.pt"]
checkpoint_inventory=[]
for name in checkpoint_files:
    p=locate(name)
    checkpoint_inventory.append({"Checkpoint":name,"Present":p is not None,
                                "Size (MiB)":round(p.stat().st_size/1048576,3) if p else None})
display(pd.DataFrame(checkpoint_inventory))

try:
    import torch
except ImportError:
    print("PyTorch is unavailable; skipping checkpoint tensor inspection.")
else:
    p=locate("best_cnn_checkpoint_compact.pt")
    if p:
        saved=torch.load(p,map_location="cpu",weights_only=True)
        state=saved.get("model_state_dict",saved) if isinstance(saved,dict) else saved
        count=sum(int(t.numel()) for t in state.values() if isinstance(t,torch.Tensor))
        print("Compact checkpoint parameter tensor elements:",count)
        if isinstance(saved,dict):
            print("Other saved metadata keys:",[k for k in saved if k!="model_state_dict"])
        if count != 26351:
            print("WARNING: checkpoint parameter count differs from the documented compact architecture.")
    else:
        print("Compact checkpoint not found; checkpoint verification deferred.")

## 9. Archived training screenshots

These screenshots document development runs and initial checks. **Smoke tests confirm that training scripts execute; they are not comparable with full-data training results.** The exported summary files remain the primary basis for reported metrics.

In [ ]:
screenshots = [
    ("Dataset verification", "emnist_verifier_results.png"),
    ("Initial CNN training", "emnist_cnn_train_result1.png"),
    ("Improved CNN training", "emnist_cnn2_train_result.png"),
    ("Compact CNN training", "emnist_compact_train_result.png"),
    ("MLP-200 training", "emnist_mlp_train_result.png"),
    ("Quantization comparison", "emnist_quantization_comparison.png"),
    ("Expanded model, first result", "emnist_epoch_100_results1.png"),
    ("Expanded model, second result", "emnist_epoch_100_results2.png"),
]
for label,filename in screenshots:
    p=locate(filename)
    if p:
        display(Markdown(f"**{label}** — `{filename}`"))
        display(Image(filename=str(p),width=780))
print("Additional smoke-test screenshots in the archive are intentionally omitted from the primary results display.")

## 10. Reproducibility and interpretation

**Evidence boundary.** This notebook records the progression from EMNIST verification and fully connected networks to reference, improved, and compact CNNs. The independent official test partition was used to report final classification accuracy, while the validation partition was used to select checkpoints. Test results from different architectures are software-reference measurements and should not be misrepresented as FPGA or Unity classification results.

**Architecture selection.** The improved CNN produced higher independent test accuracy than the compact CNN, but the compact network required fewer trainable parameters and a simpler sequence of convolution, pooling, global-average-pooling, and classification stages. This trade-off motivated its selection for hardware adaptation.

**Separately documented activities.** Integer-reference execution, fixed-point parameter export, RTL verification, Vivado resource utilization, ZedBoard camera integration, Unity deployment, inference latency, throughput, and power consumption are documented in the corresponding implementation and measurement branches, not inferred from this notebook.

**Data provenance.** The GitHub repository intentionally links to the official NIST EMNIST Balanced dataset rather than distributing the dataset. Exported JSON/CSV summaries, archived checkpoints, and screenshots provide the historical record of these completed model-development experiments. The companion `EMNIST-Models-Code` branch preserves the scripts used to generate the artifacts.

**Limitations.** Some early runs are documented only by summaries or screenshots, and full checkpoint reproducibility depends on the associated training code, compatible software environment, and dataset. Missing evidence should be identified explicitly; no substitute accuracy, checkpoint, or implementation result should be generated to fill a gap.